[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/05_access.ipynb)

# Paper 5: costing the choice between access and efficacy

Given a fixed budget, do you spend it widening access to an existing treatment or
improving how well the treatment works? A tidy uncosted answer says compare the current
levels. Put price and feasibility back in and the answer changes on about half the draws.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 20 seconds. Full run: about 25 seconds. This one is cheap either way.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here adds only a few seconds.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## Deriving $V$

Attainable capability is

$$H = \frac{(LQ)(STA)}{D\,(1 + C + \psi)} - \text{burden}.$$

Both $A$ and $T$ enter through the same product $STA$, so

$$\frac{\partial H}{\partial A} = \frac{(LQ)S\,T}{D(1+C+\psi)},
\qquad
\frac{\partial H}{\partial T} = \frac{(LQ)S\,A}{D(1+C+\psi)},
\qquad
\frac{\partial H/\partial A}{\partial H/\partial T} = \frac{T}{A}.$$

That ratio is a fact about the function, not an allocation rule. A decision maker does not
compare marginal effects; they compare gain per unit spend. An access programme buys
$\Delta_A$ at cost $c_A$, an efficacy programme buys $\Delta_T$ at cost $c_T$, so access is
the better buy exactly when

$$V = \underbrace{\frac{T}{A}}_{\text{structural}}
\cdot \underbrace{\frac{\Delta_A}{\Delta_T}}_{\text{feasibility}}
\cdot \underbrace{\frac{c_T}{c_A}}_{\text{price}} > 1 .$$

The common factor $(LQ)S/[D(1+C+\psi)]$ cancels, which is why $V$ needs none of $L$, $Q$,
$S$, $D$, $C$ or $\psi$. The uncosted rule $T > A$ is the special case where the last two
factors are both one, and there is no reason to expect that.

In [ ]:
banner()

from hpotential.core.access import AccessModel, attainable_capability, costed_statistic

high = AccessModel.high_resource()
low = AccessModel.low_resource()
pd.DataFrame({"high-resource": {k: f"{v[0]} to {v[1]}" for k, v in high.ranges.items()},
              "low-resource": {k: f"{v[0]} to {v[1]}" for k, v in low.ranges.items()}})

### Check the cancellation

The closed form drops six inputs. If that cancellation is wrong, the whole rule is wrong.
The check below builds $\partial H/\partial A$ and $\partial H/\partial T$ from the
capability expression, forms the ratio of gain per unit spend directly, and compares. The
two sides share no code, so agreement is evidence about the algebra rather than about one
implementation of it.

In [ ]:
banner()

n_check = 20_000 if FULL_RUN else 5_000
rng = np.random.default_rng(20260815)

max_rel = low.closed_form_check(rng, n_check)
print(f"maximum relative discrepancy over {n_check} draws: {max_rel:.3e}")
print(f"machine epsilon for float64:                       {np.finfo(float).eps:.3e}")
print()
print("A few multiples of machine epsilon is what an exact identity looks like in")
print("floating point. Anything at 1e-6 would mean the cancellation is approximate.")

In [ ]:
banner()

# Do the comparison by hand once, so the identity is visible rather than asserted.
p = low.draw(np.random.default_rng(1), 5)
common = (p["L"] * p["Q"]) / (p["D"] * (1.0 + p["C"] + p["psi"]))
dh_da = common * p["S"] * p["T"]
dh_dt = common * p["S"] * p["A"]
direct = (dh_da * p["dA"] / p["cA"]) / (dh_dt * p["dT"] / p["cT"])
closed = costed_statistic(p["T"], p["A"], p["dA"], p["dT"], p["cA"], p["cT"])
pd.DataFrame({"direct from dH/dA and dH/dT": direct, "closed form V": closed,
              "relative gap": np.abs(closed - direct) / direct})

## Probabilistic sensitivity analysis

Draw every input independently and uniformly over its range and ask two questions of each
draw: does the uncosted rule favour access, and does the costed rule? The interesting
number is how often they point different ways.

In [ ]:
banner()

import time

n_psa = 200_000 if FULL_RUN else 40_000
start = time.time()
rng = np.random.default_rng(20260815)
psa = {m.name: m.psa(rng, n_psa) for m in (high, low)}
print(f"{n_psa} draws per setting, {time.time() - start:.1f} s")
pd.DataFrame(psa).T[[
    "median_structural_term", "median_feasibility_term", "median_price_term", "median_V",
    "V_2_5", "V_97_5", "P_uncosted_rule_favours_access", "P_costed_rule_favours_access",
    "P_disagreement",
]].round(4)

In [ ]:
banner()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
for i, model in enumerate((high, low)):
    q = model.draw(np.random.default_rng(100 + i), min(n_psa, 60_000))
    structural = q["T"] / q["A"]
    v = structural * (q["dA"] / q["dT"]) * (q["cT"] / q["cA"])
    ax = axes[i]
    ax.hist(np.log10(v), bins=70, color=series_color(i), edgecolor="none", label="costed $V$")
    ax.hist(np.log10(structural), bins=70, color=INK, histtype="step", lw=1.0,
            label="uncosted $T/A$")
    ax.axvline(0.0, ls="--", lw=1.1, color=INK)
    ax.set_xlabel(r"$\log_{10}$ of the statistic"); ax.set_ylabel("draws")
    ax.set_title(f"{model.name}: rules disagree on "
                 f"{psa[model.name]['P_disagreement']:.1%} of draws", fontsize=10)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print("High-resource: the uncosted statistic sits mostly below one and the costed one")
print("straddles it, so the two rules split. Low-resource: both are far above one on")
print("every draw, so the shortcut costs nothing there. The disagreement is exactly")
print("zero, not approximately zero, because T/A exceeds one in every draw.")

## Interactive: three factors, one decision

The three sliders are the three factors of $V$. The decision flips live. Start at the
uncosted position, feasibility and price both at 1.0, and see what the structural term
alone says. Then move the other two and watch how little the structural term settles.

In [ ]:
banner()


def v_panel(structural=0.91, feasibility=0.67, price=1.73):
    v = structural * feasibility * price
    uncosted = structural > 1.0
    costed = v > 1.0

    fig, ax = plt.subplots(figsize=(6.6, 3.0))
    names = ["structural $T/A$", r"feasibility $\Delta_A/\Delta_T$", "price $c_T/c_A$", "$V$"]
    values = [structural, feasibility, price, v]
    colours = [series_color(0), series_color(1), series_color(2),
               series_color(2) if costed else series_color(5)]
    ax.barh(names, values, color=colours)
    ax.axvline(1.0, ls="--", lw=1.2, color=INK)
    ax.set_xlim(0, max(3.5, v * 1.15))
    ax.set_xlabel("value (the dashed line is the decision boundary at one)")
    ax.set_title(f"V = {v:.3f}: spend on {'access' if costed else 'efficacy'}", fontsize=11)
    plt.show()

    print(f"V = {structural:.3f} x {feasibility:.3f} x {price:.3f} = {v:.4f}")
    print(f"uncosted rule (T > A):   spend on {'access' if uncosted else 'efficacy'}")
    print(f"costed rule   (V > 1):   spend on {'access' if costed else 'efficacy'}")
    if uncosted != costed:
        print()
        print("The two rules disagree here. The uncosted rule is throwing away the")
        print("factors that decided it.")


explore(
    v_panel,
    controls=dict(structural=(0.2, 3.0, 0.01), feasibility=(0.1, 3.0, 0.01), price=(0.2, 4.0, 0.01)),
    defaults=dict(structural=0.91, feasibility=0.67, price=1.73),
)

---

## What this does not establish

**Every input range is illustrative.** Nothing is estimated from a health system. The
48.6 per cent disagreement rate is a property of these ranges and of the choice to draw
inputs independently. Real inputs are correlated, and a correlated prior moves the number.

**$V > 1$ ranks two programmes.** It does not say either is worth funding, and it carries no
distributional weight: a programme that helps a few people a lot and one that helps many a
little are compared only through the aggregate.

**The rule is first order.** It compares marginal gains at the current $A$ and $T$. A large
$\Delta$ takes you away from the point where the derivative was taken.

**The capability expression is a modelling choice.** The cancellation that makes $V$ so
simple depends on $A$ and $T$ entering as a product. Under a different functional form the
common factor does not cancel and $V$ is not this.

**A reduced run does not reproduce the published disagreement rate.** The closed-form check
is unaffected in substance, since it is a statement about floating-point agreement rather
than about a sample size.